# Sanity-check KSG MI on SCVI embeddings — shendure

Freshly re-compute KSG MI for SCVI on **shendure** (using the same `(X, Y)` construction as the compute script — signal one-hot-encoded) and compare against the stored **LMI** value. The stored KSG value is **not** read — we ignore it and recompute from scratch.

Signal: `author_day` (categorical → one-hot). Spot-checks use a mid-range size.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from latentmi import ksg

DATA_ROOT = Path('/home/igor/noise_scaling/data')
DATASET  = 'shendure'
ALGO     = 'SCVI'
SIGNAL   = 'author_day'
KSG_K    = 3

# full shendure grid
SIZES = [100, 359, 1291, 4641, 16681, 59948, 215443, 774263, 2782559, 10000000]
QUALITIES = [0.004, 0.0073875, 0.0136438, 0.0251984, 0.0465384,
             0.0859506, 0.1587401, 0.2931733, 0.5414548, 1.0]

# mid-range shendure size used for spot-checks and the curse-of-dim diagnostic
MID_SIZE = 16681

## Path helpers — mirror `2026-04-20_10-49_compute_ksg_scaling_curves.py`

In [2]:
def embeddings_dir(size, quality):
    return DATA_ROOT / DATASET / str(size) / str(quality) / 'results' / ALGO / 'model'

def embeddings_path(size, quality):
    return embeddings_dir(size, quality) / 'embeddings.csv'

def signal_path(quality):
    return DATA_ROOT / DATASET / 'test' / str(quality) / 'signals' / f'Y_{SIGNAL}_{quality}.csv'

def lmi_value_path(size, quality, seed=42):
    return (embeddings_dir(size, quality) / 'MI' / str(seed)
            / f'Y_{SIGNAL}_{quality}' / 'lmi_mutual_information.txt')

def load_xy(size, quality):
    """Mirror load_xy_shendure in the compute script: one-hot the single-col signal."""
    X = pd.read_csv(embeddings_path(size, quality)).values.astype(np.float64)
    sig = pd.read_csv(signal_path(quality)).iloc[:, 0].values
    Y = pd.get_dummies(sig).values.astype(np.float64)
    assert X.shape[0] == Y.shape[0]
    return X, Y

## Spot-check one configuration

shendure, mid-range size (`MID_SIZE = 16681`), quality = 1.0. Fresh KSG vs stored LMI.

In [3]:
size, quality = MID_SIZE, 1.0
X, Y = load_xy(size, quality)
print(f'X shape: {X.shape}   Y shape: {Y.shape}  (Y is one-hot over author_day)')
print(f'X dtype: {X.dtype}   Y dtype: {Y.dtype}')
print(f'X range: [{X.min():.3f}, {X.max():.3f}]   Y range: [{Y.min():.3f}, {Y.max():.3f}]')

X shape: (30000, 16)   Y shape: (30000, 43)  (Y is one-hot over author_day)
X dtype: float64   Y dtype: float64
X range: [-9.051, 6.428]   Y range: [0.000, 1.000]


In [4]:
pmi_ksg = ksg.mi(X, Y, k=KSG_K, base=2)
mi_ksg = float(np.nanmean(pmi_ksg))
print(f'Freshly computed KSG MI (bits): {mi_ksg:.4f}')

if lmi_value_path(size, quality).exists():
    stored_lmi = float(lmi_value_path(size, quality).read_text())
    print(f'Stored LMI                    : {stored_lmi:.4f} nats  ({stored_lmi/np.log(2):.4f} bits)')

KeyboardInterrupt: 

## Sweep: freshly compute KSG across all (size, quality) for shendure / SCVI

Also read back the stored LMI (seed 42) so we can plot both on the same axes. Stored KSG values are ignored.

In [ ]:
# Parallel sweep: freshly compute KSG across all (size, quality) for shendure / SCVI.
# Worker is a plain function; on Linux ProcessPoolExecutor uses fork so the child
# inherits load_xy / embeddings_path / lmi_value_path / KSG_K / np / ksg from this notebook.
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

N_WORKERS = 16  # tune to host (host has 96 CPUs); KSG releases the GIL via sklearn KDTree

def _ksg_one(args):
    size, quality = args
    emb_p = embeddings_path(size, quality)
    sig_p = signal_path(quality)
    if not emb_p.exists() or not sig_p.exists():
        return None
    try:
        X, Y = load_xy(size, quality)
    except Exception as e:
        return {'size': size, 'quality': quality, '_error': f'load: {e}'}
    n = X.shape[0]
    if n < KSG_K + 2:
        return {'size': size, 'quality': quality, '_error': f'too few ({n})'}
    pmi = ksg.mi(X, Y, k=KSG_K, base=2)
    mi_ksg = float(np.nanmean(pmi))
    lmi_p = lmi_value_path(size, quality)
    mi_lmi = float(lmi_p.read_text()) if lmi_p.exists() else float('nan')
    return {'size': size, 'quality': quality, 'n_test': n,
            'ksg_bits': mi_ksg, 'lmi_nats_stored': mi_lmi}

jobs = list(product(SIZES, QUALITIES))
records, errors = [], []
ctx = mp.get_context('fork')
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
    futures = {ex.submit(_ksg_one, j): j for j in jobs}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='configs'):
        r = fut.result()
        if r is None:
            continue
        if '_error' in r:
            errors.append(r)
            print(f"{r['size']}/{r['quality']}: {r['_error']}")
        else:
            records.append(r)

df = pd.DataFrame(records).sort_values(['size', 'quality']).reset_index(drop=True)
print(f'completed: {len(df)}  errors: {len(errors)}  workers: {N_WORKERS}')
df

configs:   0%|          | 0/100 [06:30<?, ?it/s]


## Plots

### MI vs quality, one curve per size (KSG | LMI side-by-side)

In [ ]:
# KSG (bits) vs quality, next to LMI (nats) vs quality. One line per size.
# Styling mirrors 2026-04-14_15-25_collect_and_plot_mi_scaling.ipynb:
# viridis cmap with a per-axes LogNorm over sizes, "# cells" legend on the right.
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def _fmt_size(sz):
    if sz >= 1_000_000:
        return f'{sz/1_000_000:.0f}M' if sz % 1_000_000 == 0 else f'{sz/1_000_000:.1f}M'
    if sz >= 1_000:
        return f'{sz/1_000:.0f}k' if sz % 1_000 == 0 else f'{sz/1_000:.1f}k'
    return str(sz)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharex=True)
cmap = plt.cm.viridis
sizes_sorted = sorted(df['size'].unique())
norm = mcolors.LogNorm(vmin=min(sizes_sorted), vmax=max(sizes_sorted))

for s in sizes_sorted:
    sub = df[df['size'] == s].sort_values('quality')
    color = cmap(norm(s))
    axes[0].plot(sub['quality'], sub['ksg_bits'], marker='o', markersize=4,
                 linewidth=1.2, color=color, alpha=0.9, label=_fmt_size(s))
    axes[1].plot(sub['quality'], sub['lmi_nats_stored'], marker='o', markersize=4,
                 linewidth=1.2, color=color, alpha=0.9, label=_fmt_size(s))

for ax, ylab, title in [
    (axes[0], 'KSG MI (bits)', 'KSG (freshly computed)'),
    (axes[1], 'LMI MI (nats, seed 42)', 'LMI (stored)'),
]:
    ax.set_xscale('log')
    ax.set_xlabel('Quality (downsampling ratio)', fontsize=10)
    ax.set_ylabel(ylab, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axhline(0, color='k', lw=0.4)
    ax.grid(alpha=0.25)
    ax.tick_params(labelsize=8)

axes[1].legend(title='# cells', fontsize=7, title_fontsize=8,
               loc='center left', bbox_to_anchor=(1.02, 0.5),
               frameon=True, framealpha=0.85)

fig.suptitle('shendure / SCVI — MI vs quality, one curve per size',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.93, 0.95])
plt.show()

### KSG vs LMI scatter (both in bits, with linear fit + correlation)

In [ ]:
# KSG vs LMI (both in bits) with linear fit + correlation diagnostics.
# y = slope*x + intercept, plus R^2 of that fit, Pearson r, and Spearman rho.
from scipy.stats import pearsonr, spearmanr

df_ok = df.dropna(subset=['lmi_nats_stored', 'ksg_bits']).copy()
df_ok['lmi_bits'] = df_ok['lmi_nats_stored'] / np.log(2)

x = df_ok['lmi_bits'].values
y = df_ok['ksg_bits'].values

slope, intercept = np.polyfit(x, y, 1)
y_hat = slope * x + intercept
ss_res = float(np.sum((y - y_hat) ** 2))
ss_tot = float(np.sum((y - y.mean()) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
r_p, p_p = pearsonr(x, y)
r_s, p_s = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(6.5, 6))
sc = ax.scatter(x, y, c=np.log10(df_ok['size']), cmap='viridis',
                s=40, edgecolor='k', linewidth=0.3)

x_line = np.linspace(float(x.min()), float(x.max()), 100)
fit_label = (f'fit: y = {slope:.3f} x + {intercept:.3f}\n'
             f'$R^2$ = {r2:.3f}\n'
             f'Pearson r = {r_p:.3f}  (p={p_p:.1e})\n'
             f'Spearman ρ = {r_s:.3f}  (p={p_s:.1e})')
ax.plot(x_line, slope * x_line + intercept, '-', color='C3', lw=1.6, label=fit_label)

lims = [min(x.min(), y.min()) - 0.5, max(x.max(), y.max()) + 0.5]
ax.plot(lims, lims, 'k--', lw=0.6, alpha=0.6, label='y = x')

ax.set_xlabel('LMI (bits)')
ax.set_ylabel('KSG (bits)')
ax.set_title('KSG vs LMI on identical (X, Y) — shendure / SCVI')
ax.axhline(0, color='k', lw=0.3)
ax.axvline(0, color='k', lw=0.3)
plt.colorbar(sc, label='log10(size)', ax=ax)
ax.legend(loc='upper left', fontsize=8, frameon=True, framealpha=0.85)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f'n = {len(df_ok)}   slope = {slope:.4f}   intercept = {intercept:.4f}')
print(f'R^2 = {r2:.4f}   Pearson r = {r_p:.4f} (p={p_p:.3e})   Spearman ρ = {r_s:.4f} (p={p_s:.3e})')